In [6]:
# finn_ready_model.py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU, QuantIdentity
from brevitas.export import export_qonnx
from brevitas.core.quant import QuantType

# -------------------------------
# 0. Settings / Notes
# -------------------------------
# Input will be shaped (N, 1, 1, L) to use Conv2d with kernel (1, k).
# After conv1 (kernel 8) and conv2 (kernel 4) with L=20:
#   L -> 20 - 8 + 1 = 13
#   13 -> 13 - 4 + 1 = 10
# Use MaxPool2d(kernel=(1,10)) to do fixed global pooling across the width.

# -------------------------------
# 1. Load & preprocess data
# -------------------------------
df = pd.read_csv("merged_sequences_flat_clean.csv")

# Convert sequences to float arrays
sequences = df['value_sequence'].apply(lambda x: list(map(float, x.split()))).tolist()
max_len = 20
X = np.zeros((len(sequences), max_len), dtype=np.float32)
for i, seq in enumerate(sequences):
    seq = np.array(seq[:max_len], dtype=np.float32)
    X[i, :len(seq)] = seq

y = (df['finalLabel'] == 1).astype(np.float32).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# reshape to (N, C=1, H=1, W=20)
X_train = torch.tensor(X_train[:, None, None, :])  # shape (N,1,1,20)
X_test  = torch.tensor(X_test[:, None, None, :])
y_train = torch.tensor(y_train[:, None])
y_test  = torch.tensor(y_test[:, None])

# -------------------------------
# 2. FINN-friendly quantized CNN
# -------------------------------
class FinnFriendlyQuantCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant_inp = QuantIdentity(
            bit_width=8,
            min_val=-5.0,
            max_val=5.0,
            scaling_impl_type='const',
            quant_type=QuantType.INT, # Forces integer quantization
            return_quant_tensor=True
        )
        # Use QuantConv2d with kernels shaped (1, k) for 1D sequences
        # keep bias=False (recommended for export)
        self.conv1 = QuantConv2d(in_channels=1, out_channels=8,
                                  kernel_size=(1,8), stride=(1,1),
                                  bias=False, weight_bit_width=4)
        self.act1 = QuantReLU(bit_width=4)

        self.conv2 = QuantConv2d(in_channels=8, out_channels=16,
                                  kernel_size=(1,4), stride=(1,1),
                                  bias=False, weight_bit_width=4)
        self.act2 = QuantReLU(bit_width=4)

        self.pool = nn.MaxPool2d(kernel_size=(1,10), stride=(1,10))

        self.fc = QuantLinear(16, 1, bias=False, weight_bit_width=4)

    def forward(self, x):
        x = self.quant_inp(x)      
        x = self.act1(self.conv1(x))            # -> (N,8,1,13)
        x = self.act2(self.conv2(x))            # -> (N,16,1,10)
        x = self.pool(x)                        # -> (N,16,1,1)
        x = x.view(x.size(0), -1)               # -> (N,16)
        x = self.fc(x)                          # -> (N,1)
        return x

# -------------------------------
# 3. Train
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FinnFriendlyQuantCNN().to(device)
print(model)

criterion = nn.BCEWithLogitsLoss()  # use logits
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Xtr = X_train.to(device)
Ytr = y_train.to(device)
Xte = X_test.to(device)
Yte = y_test.to(device)

for epoch in range(250):
    model.train()
    optimizer.zero_grad()
    outputs = model(Xtr)
    loss = criterion(outputs, Ytr)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            logits = model(Xte)
            preds = (torch.sigmoid(logits) > 0.5).float()
            acc = (preds.eq(Yte).sum() / Yte.shape[0]).item()
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Test Acc={acc:.4f}")

# -------------------------------
# 4. Evaluation (sklearn report)
# -------------------------------
model.eval()
with torch.no_grad():
    logits = model(Xte)
    y_pred = (torch.sigmoid(logits) > 0.5).cpu().numpy().astype(int)
    print(classification_report(Yte.cpu().numpy().astype(int), y_pred))

FinnFriendlyQuantCNN(
  (quant_inp): QuantIdentity(
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (act_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (fused_activation_quant_proxy): FusedActivationQuantProxy(
        (activation_impl): Identity()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClamp()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): ConstScaling(
            (restrict_clamp_scaling): _RestrictClampValue(
              (clamp_min_ste): ScalarClampMinSte()
              (restrict_value_impl): FloatRestrictValue()
            )
            (value): StatelessBuffer()
          )
          (int_scaling_impl): IntScaling()
          (zero_point_impl): ZeroZeroPoint(
            (zero_point)

In [9]:
# -------------------------------
# 5. Export to FINN-compatible ONNX
# -------------------------------
# Create a dummy input matching the shape used during training/export
dummy_input = torch.randn(1, 1, 1, max_len, device=device)

# Export with brevitas helper
# This will generate an ONNX using Brevitas' FINN export path
export_qonnx(model, dummy_input, "finn_ready_model.onnx")
print("Exported finn_ready_model.onnx")

Exported finn_ready_model.onnx


In [1]:
from finn.util.visualization import showInNetron
showInNetron("./finn_ready_model.onnx")

Serving './finn_ready_model.onnx' at http://0.0.0.0:8081


In [ ]:
build_dir = "."

In [3]:
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

export_onnx_path = build_dir + "/finn_ready_model.onnx"
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/finn_ready_model_tidy.onnx")

NameError: name 'build_dir' is not defined

In [9]:
from finn.util.visualization import showInNetron
showInNetron("./output/intermediate_models/step_measure_rtlsim_performance.onnx")

Serving './output/intermediate_models/step_measure_rtlsim_performance.onnx' at http://0.0.0.0:8081


In [2]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper(build_dir + "/finn_ready_model_tidy.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
# model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
# absorb final add-mul nodes into TopK
# model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(build_dir + "/finn_ready_model_streamlined.onnx")

NameError: name 'ModelWrapper' is not defined

In [ ]:
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

mfn = build_dir + "/finn_ready_model_streamlined.onnx"  # adjust path if needed
model = ModelWrapper(mfn)

# 1) conservative cleanup pass
model = model.transform(FoldConstants())
model = model.transform(RemoveUnusedTensors())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())

# 2) infer shapes & datatypes (helps partitioner reason correctly)
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())

# 3) try moving scalar linear ops, streamline again
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())

# 4) final layout inference
model = model.transform(InferDataLayouts())

# save a cleaned version to inspect
model.save(build_dir + "/finn_ready_model_cleaned_for_partition.onnx")
print("Saved cleaned model; now try CreateDataflowPartition() again")
parent_model = model.transform(CreateDataflowPartition())


In [ ]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above
fpga_part = "xc7z010clg400-1"
target_clk_ns = 10

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants


model = ModelWrapper(build_dir + "/finn_ready_model_cleaned_for_partition.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
# TopK to LabelSelect
# model = model.transform(to_hw.InferLabelSelectLayer())
# input quantization (if any) to standalone thresholding
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(to_hw.InferStreamingMaxPool())
# get rid of Reshape(-1, 1) operation between hw nodes
model = model.transform(RemoveCNVtoFCFlatten())
# get rid of Tranpose -> Tranpose identity seq
model = model.transform(absorb.AbsorbConsecutiveTransposes())
# infer tensor data layouts
model = model.transform(InferDataLayouts())
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + "/finn_ready_model_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/finn_ready_model_dataflow_model.onnx")

In [ ]:
from finn.util.visualization import showInNetron
showInNetron("./finn_ready_model_streamlined.onnx")

In [1]:
import os
    
build_dir = os.environ["FINN_BUILD_DIR"]
showInNetron(build_dir+"/end2end_cnv_w1a1_streamlined.onnx")

NameError: name 'showInNetron' is not defined

In [ ]:
from functools import partial
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.res_estimation import res_estimation

In [2]:

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveUniqueNodeNames

model = ModelWrapper(build_dir + "/finn_ready_model_dataflow_model.onnx")
model = model.transform(GiveUniqueNodeNames())

cycles_dict = model.analysis(exp_cycles_per_layer)
cycles_dict

AssertionError: File not found: /tmp/finn_dev_cutesquare/finn_ready_model_dataflow_model.onnx

In [ ]:
res_dict = model.analysis(partial(res_estimation, fpgapart="xc7z020clg400-1"))
res_dict

In [ ]:
from qonnx.custom_op.registry import getCustomOp
# Inspection script
model = ModelWrapper(build_dir + "/finn_ready_model_dataflow_model.onnx")

print("=== Model Layer Analysis ===\n")
for node in model.graph.node:
    if node.op_type in ["MVAU_hls", "ConvolutionInputGenerator_hls", 
                        "ConvolutionInputGenerator_rtl", "StreamingMaxPool_hls"]:
        print(f"\nNode: {node.name}")
        print(f"  Type: {node.op_type}")
        inst = getCustomOp(node)
        
        if "MVAU" in node.op_type:
            print(f"  MW: {inst.get_nodeattr('MW')}")
            print(f"  MH: {inst.get_nodeattr('MH')}")
        elif "ConvolutionInputGenerator" in node.op_type:
            print(f"  IFMChannels: {inst.get_nodeattr('IFMChannels')}")
            print(f"  ConvKernelDim: {inst.get_nodeattr('ConvKernelDim')}")

In [ ]:
from qonnx.custom_op.registry import getCustomOp

list_of_mvaus = model.get_nodes_by_op_type("MVAU_hls")
mvau0 = list_of_mvaus[0]

mvau0_inst = getCustomOp(mvau0)

# Get the node attributes to check the current setting
print("The parallelization parameters of %s were: " % mvau0.name)
print("PE: " + str(mvau0_inst.get_nodeattr("PE")))
print("SIMD: " + str(mvau0_inst.get_nodeattr("SIMD")))

# Set the new node attributes
mvau0_inst.set_nodeattr("PE", 4)
mvau0_inst.set_nodeattr("SIMD", 8)

# Get the node attributes to check the updated setting
print("The parallelization parameters of %s are updated to: " % mvau0.name)
print("PE: " + str(mvau0_inst.get_nodeattr("PE")))
print("SIMD: " + str(mvau0_inst.get_nodeattr("SIMD")))

In [ ]:
model.save("2dconv_PE_SIMD_modified.onnx")
showInNetron("./2dconv_PE_SIMD_modified.onnx")

In [ ]:

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveUniqueNodeNames

model = ModelWrapper(build_dir + "/2dconv_PE_SIMD_modified.onnx")
model = model.transform(GiveUniqueNodeNames())

cycles_dict = model.analysis(exp_cycles_per_layer)
cycles_dict

In [ ]:
res_dict_updated = model.analysis(partial(res_estimation, fpgapart="xc7z020clg400-1"))
res_dict_updated

In [ ]:
model_orig = ModelWrapper(build_dir + "/finn_ready_model_dataflow_model.onnx")

# Original model
list_of_mvaus = model_orig.get_nodes_by_op_type("MVAU_hls")
print("In the original model (pe=simd=1): ")
for mvau in list_of_mvaus:
    mvau_inst = getCustomOp(mvau)
    print("Layer: " + mvau.name)
    print("Input shape: " + str(mvau_inst.get_folded_input_shape()))
    print("Output shape: " + str(mvau_inst.get_folded_output_shape()))

In [ ]:

# Original model
list_of_mvaus = model.get_nodes_by_op_type("MVAU_hls")
print("In the original model (pe=simd=1): ")
for mvau in list_of_mvaus:
    mvau_inst = getCustomOp(mvau)
    print("Layer: " + mvau.name)
    print("Input shape: " + str(mvau_inst.get_folded_input_shape()))
    print("Output shape: " + str(mvau_inst.get_folded_output_shape()))

In [ ]:
# Original model
list_of_mvaus = model_orig.get_nodes_by_op_type("MVAU_hls")
print("In the original model (pe=simd=1): ")
for mvau in list_of_mvaus:
    mvau_inst = getCustomOp(mvau)
    print("Layer: " + mvau.name)
    print("Input stream width: " + str(mvau_inst.get_instream_width()))
    print("Output stream width: " + str(mvau_inst.get_outstream_width()))

In [ ]:
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
model_updated = model
model_updated = model_updated.transform(InsertDWC())
model_updated = model_updated.transform(SpecializeLayers("xc7z020clg400-1"))
model_updated = model_updated.transform(GiveUniqueNodeNames())

In [ ]:
# model_updated.save("2dconv_DWC.onnx")
from finn.util.visualization import showInNetron
showInNetron("./output/intermediate_models/step_minimize_bit_width.onnx")

In [ ]:
model_dwc = ModelWrapper("2dconv_DWC.onnx")
res_dict_dwc = model_dwc.analysis(partial(res_estimation, fpgapart="xc7z020clg400-1"))
res_dict_dwc

In [ ]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from qonnx.core.modelwrapper import ModelWrapper
from finn.util.basic import pynq_part_map
from finn.util.basic import pynq_native_port_width

build_dir = "."
# pynq_part_map["Pynq-Z1-Custom"] = "xc7z010clg400-1"

# pynq_native_port_width["Pynq-Z1-Custom"] = 64

# platform_name = "Pynq-Z1-Custom"
# target_clk_ns = 10

platform_name = "Pynq-Z1"
target_clk_ns = 10

model = ModelWrapper(build_dir+"/2dconv_DWC.onnx")
model = model.transform(ZynqBuild(platform=platform_name, period_ns=target_clk_ns))

In [ ]:
build.build_dataflow_cfg(model_file, cfg_build);